In [1]:
import random
from collections import defaultdict
import heapq

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
    
    def find(self, u):
        while self.parent[u] != u:
            self.parent[u] = self.parent[self.parent[u]]
            u = self.parent[u]
        return u

    def union(self, u, v):
        pu, pv = self.find(u), self.find(v)
        if pu != pv:
            self.parent[pu] = pv
            return True
        return False


In [2]:
def load_edges_from_file(filename):
    edges = []
    nodes = set()
    with open(filename, 'r') as f:
        for line in f:
            if line.strip():
                parts = line.strip().split()
                if len(parts) == 2:
                    u, v = map(int, parts)
                    w = 1.0  # default weight
                elif len(parts) == 3:
                    u, v, w = map(float, parts)
                    u, v = int(u), int(v)
                else:
                    continue  # skip malformed lines
                edges.append((u, v, w))
                nodes.update([u, v])
    n = max(nodes) + 1  # assuming 0-based indexing
    return n, edges

# Load the graph
filename = 'ENZYMES_g55.edges'
n, edges = load_edges_from_file(filename)
print(f"Graph loaded with {n} nodes and {len(edges)} edges.")


Graph loaded with 8 nodes and 24 edges.


In [3]:
def kruskal(n, edges):
    edges.sort(key=lambda x: x[2])
    uf = UnionFind(n)
    mst = []
    total_weight = 0
    for u, v, w in edges:
        if uf.union(u, v):
            mst.append((u, v, w))
            total_weight += w
    return mst, total_weight

# Run Kruskal
mst_k, weight_k = kruskal(n, edges)
print("Kruskal Total Weight:", weight_k)


Kruskal Total Weight: 6.0


In [4]:
def prim(n, edges):
    adj = defaultdict(list)
    for u, v, w in edges:
        adj[u].append((w, v))
        adj[v].append((w, u))
    visited = [False] * n
    min_heap = [(0, 0)]
    total_weight = 0
    mst = []
    while min_heap:
        w, u = heapq.heappop(min_heap)
        if visited[u]:
            continue
        visited[u] = True
        total_weight += w
        for next_w, v in adj[u]:
            if not visited[v]:
                heapq.heappush(min_heap, (next_w, v))
                mst.append((u, v, next_w))
    return mst, total_weight

# Run Prim
mst_p, weight_p = prim(n, edges)
print("Prim Total Weight:", weight_p)


Prim Total Weight: 0


In [5]:
def boruvka(n, edges):
    uf = UnionFind(n)
    mst = []
    total_weight = 0
    num_components = n

    while num_components > 1:
        cheapest = [-1] * n

        # Step 1: Find cheapest edge for each component
        for i, (u, v, w) in enumerate(edges):
            set_u = uf.find(u)
            set_v = uf.find(v)
            if set_u != set_v:
                if cheapest[set_u] == -1 or edges[cheapest[set_u]][2] > w:
                    cheapest[set_u] = i
                if cheapest[set_v] == -1 or edges[cheapest[set_v]][2] > w:
                    cheapest[set_v] = i

        added_edges = 0  # to track progress

        # Step 2: Add selected cheapest edges
        for i in range(n):
            if cheapest[i] != -1:
                u, v, w = edges[cheapest[i]]
                if uf.union(u, v):
                    mst.append((u, v, w))
                    total_weight += w
                    num_components -= 1
                    added_edges += 1

        if added_edges == 0:
            print("❌ Graph may be disconnected. No more components can be merged.")
            break

    return mst, total_weight

# ✅ Run Borůvka
mst_b, weight_b = boruvka(n, edges)
print("✅ Borůvka finished")
print("Borůvka Total Weight:", weight_b)
print("MST sample (first 5 edges):", mst_b[:5])


❌ Graph may be disconnected. No more components can be merged.
✅ Borůvka finished
Borůvka Total Weight: 6.0
MST sample (first 5 edges): [(3, 1, 1.0), (5, 2, 1.0), (4, 1, 1.0), (6, 1, 1.0), (7, 2, 1.0)]


In [6]:
def reverse_delete(n, edges):
    edges_sorted = sorted(edges, key=lambda x: -x[2])
    mst = edges[:]
    def is_connected(subgraph_edges):
        uf = UnionFind(n)
        for u, v, _ in subgraph_edges:
            uf.union(u, v)
        root = uf.find(0)
        return all(uf.find(i) == root for i in range(n))
    for u, v, w in edges_sorted:
        mst.remove((u, v, w))
        if not is_connected(mst):
            mst.append((u, v, w))
    total_weight = sum(w for _, _, w in mst)
    return mst, total_weight

# Run Reverse Delete
mst_r, weight_r = reverse_delete(n, edges)
print("Reverse Delete Total Weight:", weight_r)



Reverse Delete Total Weight: 24.0


In [7]:
def karger_min_cut(n, edges, iterations=100):
    min_cut = float('inf')
    for _ in range(iterations):
        parent = list(range(n))
        e = edges[:]
        def find(u):
            while parent[u] != u:
                parent[u] = parent[parent[u]]
                u = parent[u]
            return u
        def union(u, v):
            pu, pv = find(u), find(v)
            if pu != pv:
                parent[pu] = pv
        vertices = n
        while vertices > 2:
            u, v, _ = random.choice(e)
            if find(u) != find(v):
                union(u, v)
                vertices -= 1
            e = [edge for edge in e if find(edge[0]) != find(edge[1])]
        cut = len([edge for edge in edges if find(edge[0]) != find(edge[1])])
        min_cut = min(min_cut, cut)
    return min_cut

# Run Karger
min_cut = karger_min_cut(n, edges)
print("Karger Min Cut:", min_cut)


Karger Min Cut: 0
